# Dot Color Classifier

A simple CNN that learns to tell red dots from blue dots in an image.

Everything runs in this single notebook — data generation, model definition,
training, and prediction.

## 1. Imports

In [ ]:
# Install dependencies (skip if already installed)
!pip install -q torch torchvision Pillow numpy

In [ ]:
import random

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Generate synthetic data

Each image is a 64×64 noisy gray square with one colored dot.
**Class 0 = red**, **Class 1 = blue**.

In [ ]:
CLASS_COLORS = {
    0: (200, 50, 50),   # red
    1: (50, 50, 200),   # blue
}

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])


def make_dot_image(label=None, size=64):
    """Return (PIL image, label) with a colored dot on a noisy background."""
    if label is None:
        label = random.randint(0, 1)
    base = np.full((size, size, 3), 128, dtype=np.uint8)
    noise = np.random.randint(-30, 31, base.shape, dtype=np.int16)
    bg = np.clip(base.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(bg, "RGB")
    r = random.randint(4, 10)
    cx = random.randint(r + 2, size - r - 2)
    cy = random.randint(r + 2, size - r - 2)
    color = tuple(max(0, min(255, c + random.randint(-40, 40)))
                  for c in CLASS_COLORS[label])
    ImageDraw.Draw(img).ellipse([cx-r, cy-r, cx+r, cy+r], fill=color)
    return img, label


class DotDataset(Dataset):
    def __init__(self, n=1000):
        self.samples = [make_dot_image() for _ in range(n)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        img, label = self.samples[i]
        return transform(img), label


# Show a few examples
fig_imgs = [make_dot_image(label=i % 2) for i in range(6)]
for img, lbl in fig_imgs:
    print(f"label={lbl}  color={"red" if lbl == 0 else "blue"}")

## 3. Define the model

A tiny two-layer CNN.

In [ ]:
class DotClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64), nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


model = DotClassifier().to(device)
print(model)

## 4. Train

In [ ]:
N_SAMPLES = 2000
EPOCHS    = 10
BATCH     = 32
LR        = 1e-3

dataset = DotDataset(N_SAMPLES)
val_size = int(0.2 * len(dataset))
train_ds, val_ds = random_split(dataset, [len(dataset) - val_size, val_size])
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    # Validate
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            val_correct += (model(imgs).argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    print(f"Epoch {epoch:2d}/{EPOCHS}  "
          f"loss={loss_sum/total:.4f}  "
          f"train_acc={correct/total:.3f}  "
          f"val_acc={val_correct/val_total:.3f}")

## 5. Test on new images

In [ ]:
model.eval()
label_name = {0: "red", 1: "blue"}

for _ in range(8):
    img, true_label = make_dot_image()
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)
        conf, pred = probs.max(1)
    status = "✅" if pred.item() == true_label else "❌"
    print(f"{status}  true={label_name[true_label]}  "
          f"pred={label_name[pred.item()]}  conf={conf.item():.1%}")

## 6. Save / load the model (optional)

Uncomment the cells below to save or reload your trained weights.

In [ ]:
# Save
# torch.save(model.state_dict(), "model.pth")

# Load
# model = DotClassifier().to(device)
# model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
# model.eval()